In [1]:
from bs4 import BeautifulSoup
import pandas as pd

In [3]:
df = pd.read_csv('../input_data/carolina_schedule.csv', delimiter=',')
columns_to_keep = ['Day', 'Time (EDT)', 'Location', 'Event', 'Speaker']
df = df[columns_to_keep]

# Keep NaNs as NaN
# When printing or exporting, show '--' instead of NaN
print(df.to_string(na_rep='--'))


          Day   Time (EDT)                   Location                                   Event  Speaker
0      Monday  09:00–09:30           Room 1 or Lounge                        Coffee & Arrival       --
1      Monday  09:30–10:00                     Room 1                      Welcome & Overview       --
2      Monday  10:00–11:00                     Room 1                     Brief Introductions       --
3      Monday  11:00–11:30                     Room 1                   Observations Overview       --
4      Monday  11:30–12:30                     Room 1       Flash Talks: Observational Inputs       --
5      Monday  12:30–13:30                    Hallway                                   Lunch       --
6      Monday  13:30–14:30                     Room 1                   Project Pitch Block I       --
7      Monday  14:30–17:00  Room 2 / Lounge / Terrace             Discussion + Team Formation       --
8          --           --                         --                    

In [4]:
# Convert DataFrame to HTML
html_table = df.to_html(index=False, border=0, escape=False, classes='schedule-table')

# Parse the HTML with BeautifulSoup
soup = BeautifulSoup(html_table, 'html.parser')

# Find rows containing days of the week
days_of_week = ['Monday, Sept. 15',
                'Tuesday, Sept. 16',
                'Wednesday, Sept. 17',
                'Thursday, Sept. 18',
                'Friday, Sept. 19']

for day in days_of_week:
    for row in soup.find_all('tr'):
        if day in str(row):
            # Bold and center the day row and merge all its columns into one
            row_td = row.find_all('td')
            if row_td:
                merged_content = f'<td colspan="{len(row_td)}" style="text-align: center; font-weight: bold;">{day}</td>'
                row.clear()  # Clear existing td elements
                row.append(BeautifulSoup(merged_content, 'html.parser'))

# Re-convert to HTML
html_table = str(soup)

# Build the HTML content for your participants page with the table
html_content = f"""---
{html_table}
"""

print(html_content)

---
<table class="dataframe schedule-table">
<thead>
<tr style="text-align: right;">
<th>Day</th>
<th>Time (EDT)</th>
<th>Location</th>
<th>Event</th>
<th>Speaker</th>
</tr>
</thead>
<tbody>
<tr>
<td>Monday</td>
<td>09:00–09:30</td>
<td>Room 1 or Lounge</td>
<td>Coffee &amp; Arrival</td>
<td>NaN</td>
</tr>
<tr>
<td>Monday</td>
<td>09:30–10:00</td>
<td>Room 1</td>
<td>Welcome &amp; Overview</td>
<td>NaN</td>
</tr>
<tr>
<td>Monday</td>
<td>10:00–11:00</td>
<td>Room 1</td>
<td>Brief Introductions</td>
<td>NaN</td>
</tr>
<tr>
<td>Monday</td>
<td>11:00–11:30</td>
<td>Room 1</td>
<td>Observations Overview</td>
<td>NaN</td>
</tr>
<tr>
<td>Monday</td>
<td>11:30–12:30</td>
<td>Room 1</td>
<td>Flash Talks: Observational Inputs</td>
<td>NaN</td>
</tr>
<tr>
<td>Monday</td>
<td>12:30–13:30</td>
<td>Hallway</td>
<td>Lunch</td>
<td>NaN</td>
</tr>
<tr>
<td>Monday</td>
<td>13:30–14:30</td>
<td>Room 1</td>
<td>Project Pitch Block I</td>
<td>NaN</td>
</tr>
<tr>
<td>Monday</td>
<td>14:30–17:00</td>
<td>Ro

In [1]:
import pandas as pd
import math
from bs4 import BeautifulSoup

# --- Load and keep only relevant columns ---
df = pd.read_csv('../input_data/carolina_schedule.csv')
columns_to_keep = ['Day', 'Time (EDT)', 'Location', 'Event', 'Speaker']
df = df[columns_to_keep]

# --- Ensure NaNs display as '--' in HTML ---
def safe(v):
    return '--' if (v is None or (isinstance(v, float) and math.isnan(v))) else str(v)

# --- Determine track per row ---
# If your CSV already has a 'Track' column with values like 'theory', 'observations', 'sims',
# this will prefer that. Otherwise we infer from Event text.
def infer_track(event_text):
    if not isinstance(event_text, str):
        return 'default'
    t = event_text.lower()
    # Customize these keyword buckets as you like
    theory_kw = ['theory', 'theoretical', 'model', 'modelling', 'modeling', 'formalism']
    obs_kw    = ['observation', 'observational', 'imaging', 'spectro', 'survey', 'data']
    sims_kw   = ['sim', 'simulation', 'mock', 'n-body', 'hydro', 'emulator']
    if any(k in t for k in theory_kw):
        return 'theory'
    if any(k in t for k in obs_kw):
        return 'observations'
    if any(k in t for k in sims_kw):
        return 'sims'
    return 'default'

if 'Track' in df.columns:
    df['Track'] = df['Track'].fillna('').str.lower().replace('', 'default')
else:
    df['Track'] = df['Event'].apply(infer_track)

# --- Order days (use your labels exactly as in CSV) ---
day_order = [
    'Monday, Sept. 15',
    'Tuesday, Sept. 16',
    'Wednesday, Sept. 17',
    'Thursday, Sept. 18',
    'Friday, Sept. 19'
]
# Fallback: keep CSV order if day labels differ
ordered_days = [d for d in day_order if d in df['Day'].unique()] or list(df['Day'].dropna().unique())

# --- Build HTML per day with colored rows ---
table_headers = ['Time (EDT)', 'Location', 'Event', 'Speaker']
tables_html = []

for day in ordered_days:
    day_df = df[df['Day'] == day]

    # Build rows with a track class on <tr>
    rows_html = []
    for _, r in day_df.iterrows():
        tr_class = f"track-{r['Track']}"
        row_cells = ''.join(f"<td>{safe(r[col])}</td>" for col in table_headers)
        rows_html.append(f'<tr class="{tr_class}">{row_cells}</tr>')

    # Assemble a table for this day
    table_html = f"""
      <h2 class="schedule-day">{day}</h2>
      <table class="schedule-table">
        <thead>
          <tr>
            {''.join(f'<th>{h}</th>' for h in table_headers)}
          </tr>
        </thead>
        <tbody>
          {''.join(rows_html)}
        </tbody>
      </table>
    """
    tables_html.append(table_html)

# --- Final HTML snippet to paste in your Jekyll page ---
html_schedule = "\n".join(tables_html)

# Optional: print or write out to a file
print(html_schedule)
# with open('/mnt/data/schedule.html', 'w') as f:
#     f.write(html_schedule)



      <h2 class="schedule-day">Monday</h2>
      <table class="schedule-table">
        <thead>
          <tr>
            <th>Time (EDT)</th><th>Location</th><th>Event</th><th>Speaker</th>
          </tr>
        </thead>
        <tbody>
          <tr class="track-default"><td>09:00–09:30</td><td>Room 1 or Lounge</td><td>Coffee & Arrival</td><td>--</td></tr><tr class="track-default"><td>09:30–10:00</td><td>Room 1</td><td>Welcome & Overview</td><td>Sarcevic & Troxel</td></tr><tr class="track-default"><td>10:00–11:00</td><td>Room 1</td><td>Brief Introductions</td><td>all</td></tr><tr class="track-observations"><td>11:00–11:30</td><td>Room 1</td><td>Observations Overview</td><td>TBA</td></tr><tr class="track-observations"><td>11:30–12:30</td><td>Room 1</td><td>Flash Talks: Observational Inputs</td><td>--</td></tr><tr class="track-default"><td>12:30–13:30</td><td>Hallway</td><td>Lunch</td><td>--</td></tr><tr class="track-default"><td>13:30–14:30</td><td>Room 1</td><td>Project Pitch Block